In [1]:
!pip install -q torch_snippets
from torch_snippets import *
from torchvision import datasets

# define o dispositivo
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.3/110.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.4/99.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 104.8 MB/s eta 0:00:00
cpu


In [2]:
data_folder = './CIFAR10'
datasets.CIFAR10(data_folder, download=True)

100%|██████████| 170M/170M [00:02<00:00, 72.9MB/s]


Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./CIFAR10
    Split: Train

In [3]:
class Colorize(torchvision.datasets.CIFAR10):
    def __init__(self, root, train):
        super().__init__(root, train)

    def __getitem__(self, ix):
        im, _ = super().__getitem__(ix) # pega a imagem original
        bw = im.convert('L').convert('RGB')
        bw, im = np.array(bw)/255., np.array(im)/255.
        bw, im = [torch.tensor(i).permute(2,0,1).to(device).float() for i in [bw, im]]
        return bw, im  # retorna

# instancia os datasets
trn_ds = Colorize(data_folder, train=True)
val_ds = Colorize(data_folder, train=False)
trn_dl = DataLoader(trn_ds, batch_size=256, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=256, shuffle=False)

# verifica um par do dataset
bw, color = trn_ds[0]
print('Shape entrada P&B:', bw.shape, '| Shape alvo colorido:', color.shape)

Shape entrada P&B: torch.Size([3, 32, 32]) | Shape alvo colorido: torch.Size([3, 32, 32])


In [4]:
import torch.nn as nn

class Identity(nn.Module):
    def forward(self, x): return x

class DownConv(nn.Module):
    def __init__(self, ni, no, maxpool=True): # Se maxpool for True, reduz a imagem pela metade
        super().__init__()
        self.model = nn.Sequential(
            nn.MaxPool2d(2) if maxpool else Identity(),
            nn.Conv2d(ni, no, 3, padding=1),
            nn.BatchNorm2d(no),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(no, no, 3, padding=1),
            nn.BatchNorm2d(no),
            nn.LeakyReLU(0.2, inplace=True),
        )
    def forward(self, x): return self.model(x)

# ele reconstrói a resolução e combina informações
class UpConv(nn.Module):
    def __init__(self, ni, no):
        super().__init__()
        self.convtranspose = nn.ConvTranspose2d(ni, no, 2, stride=2)
        self.convlayers = nn.Sequential(
            nn.Conv2d(no+no, no, 3, padding=1),
            nn.BatchNorm2d(no),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(no, no, 3, padding=1),
            nn.BatchNorm2d(no),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x, y):
        x = self.convtranspose(x) # Sobe a resolução
        x = torch.cat([x, y], axis=1) # Conecta
        return self.convlayers(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = DownConv(3, 64, maxpool=False) # Camada inicial mantém o tamanho
        self.d2 = DownConv(64, 128)
        self.d3 = DownConv(128, 256)
        self.d4 = DownConv(256, 512)
        self.d5 = DownConv(512, 1024) # Base da UNet

        # Caminho de expansão
        self.u5 = UpConv(1024, 512)
        self.u4 = UpConv(512, 256)
        self.u3 = UpConv(256, 128)
        self.u2 = UpConv(128, 64)

        # camada final Reduz os 64 canais internos
        self.u1 = nn.Conv2d(64, 3, kernel_size=1)

    def forward(self, x):
        x0 = self.d1(x);  x1 = self.d2(x0); x2 = self.d3(x1)
        x3 = self.d4(x2); x4 = self.d5(x3)

        X4 = self.u5(x4, x3); X3 = self.u4(X4, x2)
        X2 = self.u3(X3, x1); X1 = self.u2(X2, x0)

        return self.u1(X1) # Imagem final colorida

In [6]:
# um modelod e otimizador e função
def get_model():
    model = UNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)  # adam com lr inicial
    loss_fn = nn.MSELoss()  # MSE entre RGB predito e real
    return model, optimizer, loss_fn

In [7]:
# treina um batch comparando a colorização
def train_batch(model, data, optimizer, criterion):
    model.train()
    x, y = data
    optimizer.zero_grad()
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()
    return loss.item()

# valida sem atualizar
@torch.no_grad()
def validate_batch(model, data, criterion):
    model.eval()
    x, y = data
    return criterion(model(x), y).item()

In [ ]:
model, optimizer, criterion = get_model()
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
_val_dl   = DataLoader(val_ds, batch_size=1, shuffle=True)  # loader unitário para prévia visual

# loop de treinamento
for epoch in range(100):
    for bx, data in enumerate(trn_dl):
        loss = train_batch(model, data, optimizer, criterion)

        # exibe 3 exemplos
        if (bx+1) % 50 == 0:
            for _ in range(3):
                a, b = next(iter(_val_dl))
                _b = model(a)
                subplots([a[0], b[0], _b[0]], nc=3, figsize=(5,5))

    for data in val_dl:
        validate_batch(model, data, criterion)

    scheduler.step()  # reduz o lr conforme o scheduler
    print(f'Época {epoch+1}/100 | Loss treino: {loss:.4f}')